# Car Insurance Fraud Detection
### Clean Pipeline Notebook

## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
import time
import joblib
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE, SelectFromModel
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

print('All imports successful.')

All imports successful.


## Cell 2 — Load & Preprocess Data

In [2]:
df = pd.read_csv('car_insurance.csv')
print('Loaded dataset with shape:', df.shape)

# Drop unnecessary columns
df = df.drop(columns=['incident_location'], errors='ignore')

# Normalize text and replace non-standard missing markers with NaN
replace_values = ['?', '', ' ', 'NA', 'N/A', 'na', 'n/a']
df = df.replace(replace_values, np.nan)
for c in df.select_dtypes(include=['object']).columns:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].replace({'nan': np.nan})

print('Shape after preprocessing:', df.shape)

Loaded dataset with shape: (1000, 38)
Shape after preprocessing: (1000, 37)


## Cell 3 — Feature Engineering (Date Features)

In [3]:
date_cols = []
if 'policy_bind_date' in df.columns:
    date_cols.append('policy_bind_date')
if 'incident_date' in df.columns:
    date_cols.append('incident_date')

for d in date_cols:
    df[d] = pd.to_datetime(df[d], errors='coerce')

if 'policy_bind_date' in df.columns and 'incident_date' in df.columns:
    df['policy_bind_year'] = df['policy_bind_date'].dt.year
    df['policy_bind_month'] = df['policy_bind_date'].dt.month
    df['incident_year'] = df['incident_date'].dt.year
    df['incident_month'] = df['incident_date'].dt.month
    df['days_between_incident_and_policy'] = (df['incident_date'] - df['policy_bind_date']).dt.days

if 'incident_hour_of_the_day' in df.columns:
    df['incident_hour_of_the_day'] = pd.to_numeric(df['incident_hour_of_the_day'], errors='coerce')

print('Date features added.')
print('Target distribution:')
print(df['fraud_reported'].value_counts(dropna=False))

Date features added.
Target distribution:
fraud_reported
N    753
Y    247
Name: count, dtype: int64


## Cell 4 — Prepare X and y

In [4]:
TARGET = 'fraud_reported'

if df[TARGET].dtype == object:
    df[TARGET] = df[TARGET].map({'Y': 1, 'N': 0, 'y': 1, 'n': 0}).astype(int)

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print('Numeric cols:', len(num_cols))
print('Categorical cols:', len(cat_cols))

Numeric cols: 18
Categorical cols: 17


## Cell 5 — Handle Missing Values

In [5]:
num_imputer = SimpleImputer(strategy='median')
if num_cols:
    X[num_cols] = num_imputer.fit_transform(X[num_cols])
if cat_cols:
    X[cat_cols] = X[cat_cols].fillna('Missing')

print('Missing values after imputation:', X.isnull().sum().sum())

Missing values after imputation: 0


## Cell 6 — Column Transformer / Preprocessor

In [6]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
], remainder='drop', sparse_threshold=0)

preprocessor.fit(X)
print('Preprocessor fitted.')

Preprocessor fitted.


## Cell 7 — Train/Test Split + SMOTE

In [7]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train_trans = preprocessor.transform(X_train_raw)
X_test_trans = preprocessor.transform(X_test_raw)
print('Transformed train shape:', X_train_trans.shape)
print('Transformed test shape:', X_test_trans.shape)

sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train_trans, y_train)
print('After SMOTE:', X_train_bal.shape)
print('Class distribution after SMOTE:', np.bincount(y_train_bal))

Transformed train shape: (800, 163)
Transformed test shape: (200, 163)
After SMOTE: (1204, 163)
Class distribution after SMOTE: [602 602]


## Cell 8 — Define Models & Param Grids

In [8]:
models = {
    'LogisticRegression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'RandomForestClassifier': (RandomForestClassifier(random_state=42), False),
    'XGBClassifier': (XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                                     random_state=42, tree_method='hist',
                                     objective='binary:logistic'), False),
    'SVC': (SVC(probability=True, random_state=42), True),
    'NaiveBayes': (GaussianNB(), True),
    'DecisionTreeClassifier': (DecisionTreeClassifier(random_state=42), False),
    'KNN': (KNeighborsClassifier(), True),
}

param_grids = {
    'LogisticRegression': {'C': [0.01, 0.1, 1, 10]},
    'RandomForestClassifier': {'n_estimators': [100, 200], 'max_depth': [None, 10, 20],
                                'min_samples_split': [2, 5]},
    'XGBClassifier': {'n_estimators': [100, 200], 'max_depth': [3, 6],
                       'learning_rate': [0.01, 0.1], 'subsample': [0.7, 1.0],
                       'colsample_bytree': [0.7, 1.0]},
    'SVC': {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']},
    'NaiveBayes': {},
    'DecisionTreeClassifier': {'max_depth': [None, 5, 10], 'min_samples_split': [2, 5]},
    'KNN': {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']},
}

print('Models and parameter grids ready.')

Models and parameter grids ready.


## Cell 9 — Helper Function (Train + Tune)

In [9]:
summary_results = []
saved_models = {}

def train_and_report(model_name, model_obj, needs_scaling, Xtr, ytr, Xte, yte,
                     n_iter_tune=20, cv=3, random_state=42, label=''):
    Xtr_arr, Xte_arr, scaler_local = Xtr, Xte, None

    if needs_scaling:
        scaler_local = StandardScaler()
        Xtr_arr = scaler_local.fit_transform(Xtr_arr)
        Xte_arr = scaler_local.transform(Xte_arr)

    base = clone(model_obj)
    base.fit(Xtr_arr, ytr)
    base_acc = accuracy_score(yte, base.predict(Xte_arr))
    base_f1 = f1_score(yte, base.predict(Xte_arr), average='weighted')

    tuned_acc, tuned_f1, best_est, best_params, tuning_time = base_acc, base_f1, base, None, 0

    if model_name in param_grids and param_grids[model_name]:
        rs = RandomizedSearchCV(
            estimator=clone(model_obj), param_distributions=param_grids[model_name],
            n_iter=min(n_iter_tune, max(1, len(param_grids[model_name]) * 3)),
            cv=cv, scoring='f1', n_jobs=-1, random_state=random_state, verbose=0
        )
        t0 = time.time()
        rs.fit(Xtr_arr, ytr)
        tuning_time = round(time.time() - t0, 2)
        best_est = rs.best_estimator_
        best_params = rs.best_params_
        tuned_acc = accuracy_score(yte, best_est.predict(Xte_arr))
        tuned_f1 = f1_score(yte, best_est.predict(Xte_arr), average='weighted')

    header = f'{label} {model_name}'
    print(f'--- {header} ---')
    print(f'Baseline Accuracy: {base_acc:.4f} | Baseline F1: {base_f1:.4f}')
    if best_params:
        print(f'Tuned Accuracy: {tuned_acc:.4f} | Tuned F1: {tuned_f1:.4f} | Time: {tuning_time}s')

    summary_results.append({
        'label': label, 'combination': header,
        'Baseline_Accuracy': base_acc, 'Baseline_F1': base_f1,
        'Tuned_Accuracy': tuned_acc, 'Tuned_F1': tuned_f1,
        'Tuning_Time_s': tuning_time
    })
    saved_models[header.replace(' ', '_')] = {
        'estimator': best_est, 'scaler': scaler_local, 'best_params': best_params
    }
    return best_est, scaler_local

print('Helper function ready.')

Helper function ready.


## Cell 10 — Train All Model Combinations

In [10]:
# NoFS + Logistic Regression
best_lr, scaler_lr = train_and_report(
    'LogisticRegression', *models['LogisticRegression'],
    X_train_bal, y_train_bal, X_test_trans, y_test, n_iter_tune=20, label='NoFS')

# NoFS + Random Forest
best_rf, scaler_rf = train_and_report(
    'RandomForestClassifier', *models['RandomForestClassifier'],
    X_train_bal, y_train_bal, X_test_trans, y_test, n_iter_tune=20, label='NoFS')

# SelectKBest (f_classif) + Logistic Regression
k = min(20, X_train_bal.shape[1])
skb = SelectKBest(score_func=f_classif, k=k)
skb.fit(X_train_bal, y_train_bal)
Xtr_skb, Xte_skb = skb.transform(X_train_bal), skb.transform(X_test_trans)
best_skb_lr, scaler_skb_lr = train_and_report(
    'LogisticRegression', *models['LogisticRegression'],
    Xtr_skb, y_train_bal, Xte_skb, y_test, n_iter_tune=20, label='SelectKBest')

# SelectKBest (mutual_info) + Random Forest
skb_mi = SelectKBest(score_func=mutual_info_classif, k=k)
skb_mi.fit(X_train_bal, y_train_bal)
Xtr_mi, Xte_mi = skb_mi.transform(X_train_bal), skb_mi.transform(X_test_trans)
best_mi_rf, _ = train_and_report(
    'RandomForestClassifier', *models['RandomForestClassifier'],
    Xtr_mi, y_train_bal, Xte_mi, y_test, n_iter_tune=20, label='SelectKBest_MI')

# RFE + Logistic Regression
rfe = RFE(estimator=LogisticRegression(max_iter=2000, random_state=42),
          n_features_to_select=min(20, X_train_bal.shape[1]), step=1)
rfe.fit(X_train_bal, y_train_bal)
Xtr_rfe, Xte_rfe = rfe.transform(X_train_bal), rfe.transform(X_test_trans)
best_rfe_lr, scaler_rfe_lr = train_and_report(
    'LogisticRegression', *models['LogisticRegression'],
    Xtr_rfe, y_train_bal, Xte_rfe, y_test, n_iter_tune=20, label='RFE_Logistic')

# PCA + SVC
scaler_pca = StandardScaler()
pca = PCA(n_components=min(15, X_train_bal.shape[1]), random_state=42)
Xtr_pca = pca.fit_transform(scaler_pca.fit_transform(X_train_bal))
Xte_pca = pca.transform(scaler_pca.transform(X_test_trans))
best_pca_svc, _ = train_and_report(
    'SVC', *models['SVC'], Xtr_pca, y_train_bal, Xte_pca, y_test, n_iter_tune=20, label='PCA')

# PCA + Naive Bayes
best_pca_nb, _ = train_and_report(
    'NaiveBayes', *models['NaiveBayes'], Xtr_pca, y_train_bal, Xte_pca, y_test, n_iter_tune=5, label='PCA')

# SelectFromModel + Random Forest
sfm = SelectFromModel(estimator=RandomForestClassifier(n_estimators=200, random_state=42), threshold='median')
sfm.fit(X_train_bal, y_train_bal)
Xtr_sfm, Xte_sfm = sfm.transform(X_train_bal), sfm.transform(X_test_trans)
best_sfm_rf, _ = train_and_report(
    'RandomForestClassifier', *models['RandomForestClassifier'],
    Xtr_sfm, y_train_bal, Xte_sfm, y_test, n_iter_tune=20, label='SelectFromModel_RF')

# SelectFromModel + XGBoost
best_sfm_xgb, _ = train_and_report(
    'XGBClassifier', *models['XGBClassifier'],
    Xtr_sfm, y_train_bal, Xte_sfm, y_test, n_iter_tune=40, label='SelectBest')

# NoFS + KNN
best_knn, scaler_knn = train_and_report(
    'KNN', *models['KNN'], X_train_bal, y_train_bal, X_test_trans, y_test, n_iter_tune=20, label='NoFS')

--- NoFS LogisticRegression ---
Baseline Accuracy: 0.7600 | Baseline F1: 0.7616
Tuned Accuracy: 0.8350 | Tuned F1: 0.8376 | Time: 10.64s
--- NoFS RandomForestClassifier ---
Baseline Accuracy: 0.8150 | Baseline F1: 0.8100
Tuned Accuracy: 0.8050 | Tuned F1: 0.8013 | Time: 18.79s
--- SelectKBest LogisticRegression ---
Baseline Accuracy: 0.8500 | Baseline F1: 0.8551
Tuned Accuracy: 0.8500 | Tuned F1: 0.8551 | Time: 0.15s
--- SelectKBest_MI RandomForestClassifier ---
Baseline Accuracy: 0.8050 | Baseline F1: 0.7961
Tuned Accuracy: 0.7950 | Tuned F1: 0.7912 | Time: 13.17s
--- RFE_Logistic LogisticRegression ---
Baseline Accuracy: 0.8150 | Baseline F1: 0.8179
Tuned Accuracy: 0.8250 | Tuned F1: 0.8288 | Time: 0.11s
--- PCA SVC ---
Baseline Accuracy: 0.6650 | Baseline F1: 0.6614
Tuned Accuracy: 0.6300 | Tuned F1: 0.6149 | Time: 7.71s
--- PCA NaiveBayes ---
Baseline Accuracy: 0.6250 | Baseline F1: 0.6429
--- SelectFromModel_RF RandomForestClassifier ---
Baseline Accuracy: 0.8100 | Baseline F1: 0.

OSError: [WinError -1073741795] Windows Error 0xc000001d

## Cell 11 — Leaderboard

In [11]:
summary_df = pd.DataFrame(summary_results)
summary_df = summary_df.sort_values(by='Tuned_Accuracy', ascending=False).reset_index(drop=True)
summary_df.to_csv('fs_model_leaderboard.csv', index=False)
print('Leaderboard saved.')
summary_df.head(20)

Leaderboard saved.


,label,combination,Baseline_Accuracy,Baseline_F1,Tuned_Accuracy,Tuned_F1,Tuning_Time_s
0,SelectFromModel_RF,SelectFromModel_RF RandomForestClassifier,0.810,0.808646,0.855,0.859576,18.52
1,SelectKBest,SelectKBest LogisticRegression,0.850,0.855078,0.850,0.855078,0.15
2,NoFS,NoFS LogisticRegression,0.760,0.761600,0.835,0.837616,10.64
3,RFE_Logistic,RFE_Logistic LogisticRegression,0.815,0.817933,0.825,0.828757,0.11
4,NoFS,NoFS RandomForestClassifier,0.815,0.809980,0.805,0.801346,18.79
5,SelectKBest_MI,SelectKBest_MI RandomForestClassifier,0.805,0.796099,0.795,0.791159,13.17
6,PCA,PCA SVC,0.665,0.661358,0.630,0.614907,7.71
7,PCA,PCA NaiveBayes,0.625,0.642860,0.625,0.642860,0.00


## Cell 12 — Save Best Model + All Artifacts

In [12]:
top = summary_df.iloc[0]
print('Best combo:', top['combination'], '| Tuned Accuracy:', top['Tuned_Accuracy'])

key = top['combination'].replace(' ', '_')
best_record = saved_models.get(key)

if best_record is not None:
    estimator = best_record['estimator']
    scaler_local = best_record['scaler']
    
    print('Model type:', type(estimator))
    print('Needs scaling:', scaler_local is not None)

    # Build pipeline based on whether scaling is needed
    if scaler_local is not None:
        full_pipeline = Pipeline(steps=[
            ('selector', sfm),           # SelectFromModel
            ('scaler', scaler_local),     # only if needed
            ('model', estimator)
        ])
    else:
        full_pipeline = Pipeline(steps=[
            ('selector', sfm),           # SelectFromModel
            ('model', estimator)          # no scaler needed
        ])

    joblib.dump(full_pipeline, 'full_pipeline.pkl')
    joblib.dump(preprocessor, 'preprocessor_pipeline.pkl')
    joblib.dump(num_imputer, 'num_imputer.pkl')
    print('All artifacts saved correctly!')
    print('Pipeline steps:', [(name, type(step)) for name, step in full_pipeline.steps])
else:
    print('Best estimator not found. Available keys:')
    print(list(saved_models.keys()))

Best combo: SelectFromModel_RF RandomForestClassifier | Tuned Accuracy: 0.855
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Needs scaling: False
All artifacts saved correctly!
Pipeline steps: [('selector', <class 'sklearn.feature_selection._from_model.SelectFromModel'>), ('model', <class 'sklearn.ensemble._forest.RandomForestClassifier'>)]


## Cell 13 — Prediction Pipeline

In [13]:
import joblib
import pandas as pd

# Load artifacts
full_pipeline = joblib.load('full_pipeline.pkl')
imputer = joblib.load('num_imputer.pkl')
preprocessor = joblib.load('preprocessor_pipeline.pkl')

def eng_features(df):
    df = df.copy()
    date_cols = []
    if 'policy_bind_date' in df.columns:
        date_cols.append('policy_bind_date')
    if 'incident_date' in df.columns:
        date_cols.append('incident_date')
    for d in date_cols:
        df[d] = pd.to_datetime(df[d], errors='coerce')
    if 'policy_bind_date' in df.columns and 'incident_date' in df.columns:
        df['policy_bind_year'] = df['policy_bind_date'].dt.year
        df['policy_bind_month'] = df['policy_bind_date'].dt.month
        df['incident_year'] = df['incident_date'].dt.year
        df['incident_month'] = df['incident_date'].dt.month
        df['days_between_incident_and_policy'] = (
            df['incident_date'] - df['policy_bind_date']
        ).dt.days
    if 'incident_hour_of_the_day' in df.columns:
        df['incident_hour_of_the_day'] = pd.to_numeric(
            df['incident_hour_of_the_day'], errors='coerce')
    return df

def predict_insurance(input_df):
    df = eng_features(input_df)
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    df[num_cols] = imputer.transform(df[num_cols])
    X_processed = preprocessor.transform(df)
    prediction = full_pipeline.predict(X_processed)
    probability = full_pipeline.predict_proba(X_processed)
    return prediction, probability

print('Prediction pipeline ready.')

Prediction pipeline ready.


## Cell 14 — Quick Test

In [14]:
df_test = pd.read_csv('car_insurance.csv')
sample = df_test.drop(columns=['fraud_reported']).iloc[[0]]

prediction, probability = predict_insurance(sample)
print(f'Prediction: {prediction[0]} (1=Fraud, 0=No Fraud)')
print(f'Fraud Probability: {probability[0][1]:.2%}')

Prediction: 1 (1=Fraud, 0=No Fraud)
Fraud Probability: 72.57%
